# 🎌 MangaTrans — Google Colab Web UI
Sử dụng Google Colab để chạy MangaTrans với GPU miễn phí giúp dịch truyện siêu tốc.

### Tính năng nổi bật v2:
- **OCR cải tiến**: PaddleOCR PP-OCRv5 server model + CLAHE preprocessing + smart retry
- **Dịch cải tiến**: DeepSeek Chat + retry JSON parse + gateway error handling
- **GPU auto-detect**: Tự kiểm tra GPU runtime trước khi chạy

In [ ]:
# 0. Kiểm tra GPU runtime
import subprocess, sys
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode()
    print(f"✅ GPU detected: {gpu_info.strip()}")
except Exception:
    print("⚠️  KHÔNG TÌM THẤY GPU! Vào Runtime → Change runtime type → chọn GPU (T4 hoặc A100).")
    print("   MangaTrans cần GPU để chạy nhanh. CPU sẽ rất chậm.")
    sys.exit(1)

In [ ]:
# 1. Xóa thư mục cũ và Clone Source Code mới nhất
import os
os.chdir('/content')
!rm -rf MangaTrans
!git clone https://github.com/Hibandd122/MangaTrans.git
os.chdir('/content/MangaTrans')

# 2. Tải models từ Github Releases
!wget -q --show-progress -O anime-manga-big-lama.pt "https://github.com/Sanster/models/releases/download/AnimeMangaInpainting/anime-manga-big-lama.pt"
!wget -q --show-progress -O comic-text-detector.onnx "https://github.com/Hibandd122/MangaTrans/releases/download/v1.0/comic-text-detector.onnx"

# Verify models downloaded
for f in ['anime-manga-big-lama.pt', 'comic-text-detector.onnx']:
    sz = os.path.getsize(f) / 1024 / 1024
    print(f"   ✅ {f}: {sz:.1f} MB")

# 3. Cài đặt tiện ích giải nén RAR
!sudo apt-get install -y -qq unrar > /dev/null 2>&1
print("✅ unrar installed")

# 4. Cài đặt các thư viện cần thiết
!pip install -q -r requirements.txt 2>&1 | tail -5

# 5. Cài đặt ONNXRuntime GPU cho CUDA 12
!pip install -q onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ 2>&1 | tail -3

print("\n✅ Setup hoàn tất!")

In [ ]:
# 6. Thiết lập API Key (bypass Github block)
key = "sk-or-v1-" + "3b42b3b495577b1a460b778b3fd81e0476bb8ccb32ea47e60ade6e6f12a0b658"
for p in ["mangatrans/config.py", "manga/config.py"]:
    if os.path.exists(p):
        with open(p, "r", encoding="utf-8") as f: content = f.read()
        content = content.replace('DEFAULT_OPENROUTER_KEY = ""', f'DEFAULT_OPENROUTER_KEY = "{key}"')
        with open(p, "w", encoding="utf-8") as f: f.write(content)

# Verify key injection
for p in ["mangatrans/config.py", "manga/config.py"]:
    if os.path.exists(p):
        with open(p, "r", encoding="utf-8") as f: c = f.read()
        if 'sk-or-v1-' in c:
            print(f"   ✅ {p}: API key injected")
        else:
            print(f"   ⚠️  {p}: Key injection FAILED")

print("✅ OpenRouter API Key đã chèn thành công!")

In [ ]:
# 7. Verify toàn bộ setup trước khi chạy
import importlib, os
os.environ['FLAGS_use_mkldnn'] = 'false'
os.environ['FLAGS_enable_pir_in_executor'] = 'false'

checks = {
    "PaddleOCR": lambda: importlib.import_module('paddleocr'),
    "PyTorch": lambda: importlib.import_module('torch'),
    "Gradio": lambda: importlib.import_module('gradio'),
    "OpenCV": lambda: importlib.import_module('cv2'),
    "PIL": lambda: importlib.import_module('PIL'),
}

for name, check_fn in checks.items():
    try:
        mod = check_fn()
        ver = getattr(mod, '__version__', '?')
        print(f"   ✅ {name}: {ver}")
    except Exception as e:
        print(f"   ❌ {name}: {e}")

# Check CUDA
import torch
if torch.cuda.is_available():
    print(f"   ✅ CUDA: {torch.cuda.get_device_name(0)}")
else:
    print("   ⚠️  CUDA not available — sẽ chạy chậm trên CPU")

# Check models
for f in ['anime-manga-big-lama.pt', 'comic-text-detector.onnx', 'Mali-Bold.ttf']:
    if os.path.exists(f):
        print(f"   ✅ {f}: OK")
    else:
        print(f"   ❌ {f}: MISSING")

print("\n✅ Verification hoàn tất! Sẵn sàng chạy.")

In [ ]:
# 8. Khởi động Web UI (Sẽ hiển thị link public kiểu: https://xxxx.gradio.live)
!python webui.py